In [ ]:
!pip install -q -U \
    langchain \
    langchain-core \
    langchain-chroma \
    langchain-huggingface \
    langchain-community \
    sentence-transformers \
    chromadb \
    rank-bm25 \
    pandas \
    openpyxl

In [ ]:
# ==========================================================
# IMPORT LIBRARIES
# ==========================================================

import os
import gc
import re
import torch
import pandas as pd

from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.retrievers import BM25Retriever

print("Imports successful.")

In [ ]:
# ==========================================================
# MOUNT GOOGLE DRIVE
# ==========================================================

from google.colab import drive

drive.mount('/content/drive')

In [ ]:
# ==========================================================
# PROJECT PATHS
# ==========================================================

PROJECT_DIR = "/content/drive/MyDrive/RAG_ICH"

DATA_DIR = os.path.join(PROJECT_DIR, "data")

CHUNK_DIR = os.path.join(
    DATA_DIR,
    "chunks"
)

EVALUATION_DIR = os.path.join(
    DATA_DIR,
    "evaluation"
)

VECTOR_DIR = os.path.join(
    PROJECT_DIR,
    "vector_db"
)

RESULT_DIR = os.path.join(
    PROJECT_DIR,
    "results",
    "retrieval"
)

# Create folders if they do not exist
os.makedirs(VECTOR_DIR, exist_ok=True)
os.makedirs(RESULT_DIR, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Results directory:", RESULT_DIR)

In [ ]:
# ==========================================================
# EXPERIMENT CONFIGURATION
# ==========================================================

# Fixed chunking strategy
CHUNK_FILE = "recursive_chunks.csv"

CHUNK_PATH = os.path.join(
    CHUNK_DIR,
    CHUNK_FILE
)

# Evaluation question file
EVAL_FILE = os.path.join(
    EVALUATION_DIR,
    "evaluation_questions.xlsx"
)

# Selected embedding model
EMBEDDING_MODEL_NAME = (
    "FremyCompany/BioLORD-2023-C"
)

# Retrieval settings
TOP_K = 5

# RRF fusion constant
RRF_K = 60

print("Chunk file:", CHUNK_PATH)
print("Evaluation file:", EVAL_FILE)
print("Embedding model:", EMBEDDING_MODEL_NAME)
print("Top-k:", TOP_K)
print("RRF constant:", RRF_K)

In [ ]:
# ==========================================================
# CHECK FILES
# ==========================================================

print(
    "Recursive chunk file exists:",
    os.path.exists(CHUNK_PATH)
)

print(
    "Evaluation file exists:",
    os.path.exists(EVAL_FILE)
)

In [ ]:
# ==========================================================
# LOAD RECURSIVE CHUNKS
# ==========================================================

chunk_df = pd.read_csv(
    CHUNK_PATH
)

print(
    f"Number of recursive chunks: {len(chunk_df)}"
)

print("\nColumns:")
print(
    chunk_df.columns.tolist()
)

display(
    chunk_df.head()
)

In [ ]:
# ==========================================================
# CREATE LANGCHAIN DOCUMENTS
# ==========================================================

documents = []

for _, row in chunk_df.iterrows():

    doc = Document(

        page_content=str(row["text"]),

        metadata={

            "chunk_id": int(
                row["chunk_id"]
            ),

            "chunking_method":
                "recursive"

        }

    )

    documents.append(doc)

print(
    f"Created {len(documents)} documents."
)

In [ ]:
# ==========================================================
# LOAD EVALUATION QUESTIONS
# ==========================================================

evaluation_df = pd.read_excel(
    EVAL_FILE
)

print(
    f"Number of evaluation questions: "
    f"{len(evaluation_df)}"
)

print("\nColumns:")

print(
    evaluation_df.columns.tolist()
)

display(
    evaluation_df.head()
)

In [ ]:
# ==========================================================
# CREATE BM25 RETRIEVER
# ==========================================================

bm25_retriever = BM25Retriever.from_documents(
    documents
)

bm25_retriever.k = TOP_K

print(
    f"BM25 retriever created with k={TOP_K}"
)

In [ ]:
# ==========================================================
# CHECK GPU / DEFINE DEVICE
# ==========================================================

print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    DEVICE = "cuda"
    print("Using GPU:", torch.cuda.get_device_name(0))
else:
    DEVICE = "cpu"
    print("Using CPU")

In [ ]:
# ==========================================================
# LOAD BIOLORD EMBEDDING MODEL
# ==========================================================

embedding_model = HuggingFaceEmbeddings(

    model_name=
        EMBEDDING_MODEL_NAME,

    model_kwargs={
        "device": DEVICE
    },

    encode_kwargs={
        "normalize_embeddings": True
    }

)

print(
    "BioLORD embedding model loaded successfully."
)

In [ ]:
# ==========================================================
# CREATE DENSE VECTOR DATABASE
# ==========================================================

DENSE_VECTOR_DIR = os.path.join(
    VECTOR_DIR,
    "retrieval_evaluation",
    "biolord_dense"
)

os.makedirs(
    DENSE_VECTOR_DIR,
    exist_ok=True
)

COLLECTION_NAME = (
    "recursive_biolord_retrieval"
)

print(
    "Creating ChromaDB..."
)

vector_db = Chroma.from_documents(

    documents=documents,

    embedding=embedding_model,

    collection_name=COLLECTION_NAME,

    persist_directory=DENSE_VECTOR_DIR

)

print(
    "Dense ChromaDB created successfully."
)

In [ ]:
# ==========================================================
# CREATE DENSE RETRIEVER
# ==========================================================

dense_retriever = vector_db.as_retriever(

    search_type="similarity",

    search_kwargs={
        "k": TOP_K
    }

)

print(
    f"Dense retriever created with k={TOP_K}"
)

In [ ]:
# ==========================================================
# TEST RETRIEVAL
# ==========================================================

test_question = (
    evaluation_df.iloc[0]["clinical_question"]
)

print("QUESTION:")
print(test_question)

print("\n" + "=" * 70)
print("BM25 RESULTS")
print("=" * 70)

bm25_docs = bm25_retriever.invoke(
    test_question
)

for rank, doc in enumerate(
    bm25_docs,
    start=1
):

    print(
        f"\nRank {rank}"
    )

    print(
        f"Chunk ID: "
        f"{doc.metadata['chunk_id']}"
    )

    print(
        doc.page_content[:500]
    )


print("\n" + "=" * 70)
print("DENSE RESULTS")
print("=" * 70)

dense_docs = dense_retriever.invoke(
    test_question
)

for rank, doc in enumerate(
    dense_docs,
    start=1
):

    print(
        f"\nRank {rank}"
    )

    print(
        f"Chunk ID: "
        f"{doc.metadata['chunk_id']}"
    )

    print(
        doc.page_content[:500]
    )

In [ ]:
# ==========================================================
# RECIPROCAL RANK FUSION (RRF)
# ==========================================================

def reciprocal_rank_fusion(
    bm25_docs,
    dense_docs,
    rrf_k=60,
    top_k=5
):

    rrf_scores = {}

    document_lookup = {}

    # ------------------------------------------------------
    # BM25 RANKING
    # ------------------------------------------------------

    for rank, doc in enumerate(
        bm25_docs,
        start=1
    ):

        chunk_id = (
            doc.metadata["chunk_id"]
        )

        document_lookup[
            chunk_id
        ] = doc

        if chunk_id not in rrf_scores:

            rrf_scores[
                chunk_id
            ] = 0

        rrf_scores[
            chunk_id
        ] += 1 / (
            rrf_k + rank
        )

    # ------------------------------------------------------
    # DENSE RANKING
    # ------------------------------------------------------

    for rank, doc in enumerate(
        dense_docs,
        start=1
    ):

        chunk_id = (
            doc.metadata["chunk_id"]
        )

        document_lookup[
            chunk_id
        ] = doc

        if chunk_id not in rrf_scores:

            rrf_scores[
                chunk_id
            ] = 0

        rrf_scores[
            chunk_id
        ] += 1 / (
            rrf_k + rank
        )

    # ------------------------------------------------------
    # SORT BY RRF SCORE
    # ------------------------------------------------------

    ranked_chunk_ids = sorted(

        rrf_scores,

        key=rrf_scores.get,

        reverse=True

    )

    # ------------------------------------------------------
    # RETURN TOP-K DOCUMENTS
    # ------------------------------------------------------

    hybrid_docs = [

        document_lookup[chunk_id]

        for chunk_id
        in ranked_chunk_ids[:top_k]

    ]

    return hybrid_docs

In [ ]:
# ==========================================================
# TEST HYBRID RETRIEVAL
# ==========================================================

test_question = (
    evaluation_df.iloc[0]["clinical_question"]
)

bm25_docs = bm25_retriever.invoke(
    test_question
)

dense_docs = dense_retriever.invoke(
    test_question
)

hybrid_docs = reciprocal_rank_fusion(

    bm25_docs=bm25_docs,

    dense_docs=dense_docs,

    rrf_k=RRF_K,

    top_k=TOP_K

)

print(
    "HYBRID RETRIEVAL RESULTS"
)

for rank, doc in enumerate(
    hybrid_docs,
    start=1
):

    print("\n" + "-" * 50)

    print(
        f"Rank {rank}"
    )

    print(
        f"Chunk ID: "
        f"{doc.metadata['chunk_id']}"
    )

    print(
        doc.page_content[:500]
    )

In [ ]:
# ==========================================================
# CREATE RETRIEVAL RESULT FUNCTION
# ==========================================================

def create_result_row(
    row,
    rank,
    doc,
    retrieval_method
):

    return {

        "question_id":
            row["question_id"],

        "domain":
            row["domain"],

        "subdomain":
            row["subdomain"],

        "clinical_question":
            row["clinical_question"],

        "reference_answer_draft":
            row["reference_answer_draft"],

        "gold_evidence_location":
            row["gold_evidence_location"],

        "required_gold_evidence_for_retrieval":
            row[
                "required_gold_evidence_for_retrieval"
            ],

        "retrieval_method":
            retrieval_method,

        "rank":
            rank,

        "chunk_id":
            doc.metadata["chunk_id"],

        "retrieved_text":
            doc.page_content

    }

In [ ]:
# ==========================================================
# RUN BM25 FOR ALL QUESTIONS
# ==========================================================

bm25_results = []

print(
    "Running BM25 retrieval..."
)

for _, row in evaluation_df.iterrows():

    question = (
        row["clinical_question"]
    )

    retrieved_docs = (
        bm25_retriever.invoke(
            question
        )
    )

    for rank, doc in enumerate(
        retrieved_docs,
        start=1
    ):

        result = create_result_row(

            row=row,

            rank=rank,

            doc=doc,

            retrieval_method="BM25"

        )

        bm25_results.append(
            result
        )

bm25_results_df = pd.DataFrame(
    bm25_results
)

print(
    f"BM25 retrieval completed."
)

print(
    f"Total rows: "
    f"{len(bm25_results_df)}"
)

display(
    bm25_results_df.head()
)

In [ ]:
# ==========================================================
# SAVE BM25 RESULTS
# ==========================================================

BM25_OUTPUT = os.path.join(

    RESULT_DIR,

    "bm25_retrieval_results.csv"

)

bm25_results_df.to_csv(

    BM25_OUTPUT,

    index=False

)

print(
    "BM25 results saved:"
)

print(
    BM25_OUTPUT
)

In [ ]:
# ==========================================================
# RUN DENSE RETRIEVAL FOR ALL QUESTIONS
# ==========================================================

dense_results = []

print(
    "Running dense retrieval..."
)

for _, row in evaluation_df.iterrows():

    question = (
        row["clinical_question"]
    )

    retrieved_docs = (
        dense_retriever.invoke(
            question
        )
    )

    for rank, doc in enumerate(
        retrieved_docs,
        start=1
    ):

        result = create_result_row(

            row=row,

            rank=rank,

            doc=doc,

            retrieval_method="Dense"

        )

        dense_results.append(
            result
        )

dense_results_df = pd.DataFrame(
    dense_results
)

print(
    "Dense retrieval completed."
)

print(
    f"Total rows: "
    f"{len(dense_results_df)}"
)

display(
    dense_results_df.head()
)

In [ ]:
# ==========================================================
# SAVE DENSE RETRIEVAL RESULTS
# ==========================================================

DENSE_OUTPUT = os.path.join(

    RESULT_DIR,

    "dense_retrieval_results.csv"

)

dense_results_df.to_csv(

    DENSE_OUTPUT,

    index=False

)

print(
    "Dense retrieval results saved:"
)

print(
    DENSE_OUTPUT
)

In [ ]:
# ==========================================================
# RUN HYBRID RETRIEVAL FOR ALL QUESTIONS
# ==========================================================

hybrid_results = []

print(
    "Running hybrid retrieval..."
)

for _, row in evaluation_df.iterrows():

    question = (
        row["clinical_question"]
    )

    # BM25 retrieval
    bm25_docs = (
        bm25_retriever.invoke(
            question
        )
    )

    # Dense retrieval
    dense_docs = (
        dense_retriever.invoke(
            question
        )
    )

    # RRF fusion
    hybrid_docs = (
        reciprocal_rank_fusion(

            bm25_docs=bm25_docs,

            dense_docs=dense_docs,

            rrf_k=RRF_K,

            top_k=TOP_K

        )
    )

    # Save results
    for rank, doc in enumerate(
        hybrid_docs,
        start=1
    ):

        result = create_result_row(

            row=row,

            rank=rank,

            doc=doc,

            retrieval_method="Hybrid_RRF"

        )

        hybrid_results.append(
            result
        )

hybrid_results_df = pd.DataFrame(
    hybrid_results
)

print(
    "Hybrid retrieval completed."
)

print(
    f"Total rows: "
    f"{len(hybrid_results_df)}"
)

display(
    hybrid_results_df.head()
)

In [ ]:
# ==========================================================
# SAVE HYBRID RETRIEVAL RESULTS
# ==========================================================

HYBRID_OUTPUT = os.path.join(

    RESULT_DIR,

    "hybrid_retrieval_results.csv"

)

hybrid_results_df.to_csv(

    HYBRID_OUTPUT,

    index=False

)

print(
    "Hybrid retrieval results saved:"
)

print(
    HYBRID_OUTPUT
)

In [ ]:
# ==========================================================
# VERIFY RESULTS
# ==========================================================

print("=" * 60)

print("BM25")

print(
    f"Rows: "
    f"{len(bm25_results_df)}"
)

print(
    f"File exists: "
    f"{os.path.exists(BM25_OUTPUT)}"
)

print("\n" + "=" * 60)

print("DENSE")

print(
    f"Rows: "
    f"{len(dense_results_df)}"
)

print(
    f"File exists: "
    f"{os.path.exists(DENSE_OUTPUT)}"
)

print("\n" + "=" * 60)

print("HYBRID")

print(
    f"Rows: "
    f"{len(hybrid_results_df)}"
)

print(
    f"File exists: "
    f"{os.path.exists(HYBRID_OUTPUT)}"
)

In [ ]:
# ==========================================================
# COMBINE ALL RETRIEVAL RESULTS
# ==========================================================

combined_results_df = pd.concat(

    [
        bm25_results_df,
        dense_results_df,
        hybrid_results_df
    ],

    ignore_index=True

)

COMBINED_OUTPUT = os.path.join(

    RESULT_DIR,

    "all_retrieval_methods_results.csv"

)

combined_results_df.to_csv(

    COMBINED_OUTPUT,

    index=False

)

print(
    f"Total combined rows: "
    f"{len(combined_results_df)}"
)

print(
    "Combined results saved:"
)

print(
    COMBINED_OUTPUT
)